In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import gzip
import urllib.request as request
import os
import torch.optim as optim
import torchvision.datasets as datasets
import h5py
import time
import pandas as pd
import random
import math
from copy import deepcopy
from torch.optim import Adam
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
from torch.utils.data import Subset
from torchmetrics.image import StructuralSimilarityIndexMeasure
from pytorch_msssim import ssim
from IPython.display import display

In [2]:
SEED = 42
FILE_PATH = 'Diffusion_jets_procesados_por_canal.hdf5'
CHANNEL_NAMES = ['TRACKER', 'ECAL', 'HCAL']
NORMALIZATION_PERCENTILE = 99.5
SPLIT_SEED = 42
USE_SMALL_SUBSET = True
TRAIN_SMALL_SIZE = 10000
VAL_SMALL_SIZE = 2500
TEST_SMALL_SIZE = 2500
BATCH_SIZE = 16
NUM_EPOCHS = 40
LEARNING_RATE = 0.0002
LATENT_CHANNELS = 4
BASE_CHANNELS = 64
BETA_MAX = 0.001
KL_START_EPOCH = 5
KL_WARMUP_EPOCHS = 15
ACTIVE_THRESHOLD_01 = 0.0001
ACTIVE_WEIGHT = 5.0
BACKGROUND_WEIGHT = 1.0
ENERGY_WEIGHT = 1.0
BACKGROUND_ENERGY_WEIGHT = 0.01
OCCUPANCY_WEIGHT = 2.0
FALSE_POSITIVE_WEIGHT = 0.5
PEAK_WEIGHT = 1.0
GATE_TEMPERATURE = 0.05
TOPK_PEAK_PIXELS = 8
MASK_WEIGHT = 0.1
OCCUPANCY_THRESHOLD = 0.8
TRACKER_OCCUPANCY_THRESHOLD = 0.79202
ECAL_OCCUPANCY_THRESHOLD = 0.79697
HCAL_OCCUPANCY_THRESHOLD = 0.826667


def get_occupancy_thresholds(reference_tensor):
    return reference_tensor.new_tensor([TRACKER_OCCUPANCY_THRESHOLD, ECAL_OCCUPANCY_THRESHOLD, HCAL_OCCUPANCY_THRESHOLD]).view(1, 3, 1, 1)

In [3]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: mps


In [4]:
def raw_to_01(x):
    if x.min() < 0:
        x = (x + 1.0) / 2.0

    return np.clip(x, 0.0, None)


def make_split_indices(total_size, train_fraction=0.8, val_fraction=0.1, seed=42):
    train_size = int(train_fraction * total_size)
    val_size = int(val_fraction * total_size)
    generator = torch.Generator().manual_seed(seed)
    permutation = torch.randperm(total_size, generator=generator).numpy()
    train_indices = permutation[:train_size]
    val_indices = permutation[train_size:train_size + val_size]
    test_indices = permutation[train_size + val_size:]

    return (train_indices, val_indices, test_indices)


def compute_global_active_scale(file_path, train_indices, percentile=99.5, chunk_size=512):
    sorted_indices = np.sort(np.asarray(train_indices, dtype=np.int64))
    active_chunks = []

    with h5py.File(file_path, 'r') as file:
        dataset = file['X_diffusion']

        for start in range(0, len(sorted_indices), chunk_size):
            batch_indices = sorted_indices[start:start + chunk_size]
            x = dataset[batch_indices].astype(np.float32)
            x_01 = raw_to_01(x)
            active = x_01[x_01 > 0.0]

            if active.size:
                active_chunks.append(active)

    if not active_chunks:
        return 1.0

    active_values = np.concatenate(active_chunks)

    return float(np.percentile(active_values, percentile))

In [5]:
class IndexedJetsDataset(Dataset):

    def __init__(self, file_path, indices, global_scale):
        super().__init__()
        self.indices = np.asarray(indices, dtype=np.int64)
        self.global_scale = float(global_scale)

        if self.global_scale <= 0:
            raise ValueError('global_scale debe ser positiva.')

        order = np.argsort(self.indices)
        sorted_indices = self.indices[order]

        with h5py.File(file_path, 'r') as file:
            x_sorted = file['X_diffusion'][sorted_indices].astype(np.float32)
            y_sorted = file['y'][sorted_indices].astype(np.int64)

        inverse_order = np.argsort(order)
        x = x_sorted[inverse_order]
        self.y = y_sorted[inverse_order]
        x_01 = raw_to_01(x)
        x_01 = np.clip(x_01 / (self.global_scale + 1e-08), 0.0, 1.0)
        self.x = (2.0 * x_01 - 1.0).astype(np.float32)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, index):
        image = torch.from_numpy(self.x[index]).permute(2, 0, 1).float()
        label = torch.tensor(self.y[index], dtype=torch.long)

        return (image, label)

In [6]:
with h5py.File(FILE_PATH, 'r') as file:
    total_size = file['X_diffusion'].shape[0]

train_indices_full, val_indices_full, test_indices_full = make_split_indices(total_size=total_size, seed=SPLIT_SEED)

if USE_SMALL_SUBSET:
    train_indices = train_indices_full[:TRAIN_SMALL_SIZE]
    val_indices = val_indices_full[:VAL_SMALL_SIZE]
    test_indices = test_indices_full[:TEST_SMALL_SIZE]
else:
    train_indices = train_indices_full
    val_indices = val_indices_full
    test_indices = test_indices_full

global_train_scale = compute_global_active_scale(file_path=FILE_PATH, train_indices=train_indices, percentile=NORMALIZATION_PERCENTILE)
train_dataset = IndexedJetsDataset(FILE_PATH, train_indices, global_train_scale)
val_dataset = IndexedJetsDataset(FILE_PATH, val_indices, global_train_scale)
test_dataset = IndexedJetsDataset(FILE_PATH, test_indices, global_train_scale)
train_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, generator=train_generator)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print('Total:', total_size)
print('Train:', len(train_dataset))
print('Validation:', len(val_dataset))
print('Test:', len(test_dataset))
print(f'Escala global P{NORMALIZATION_PERCENTILE}: {global_train_scale:.8f}')

Total: 139306
Train: 10000
Validation: 2500
Test: 2500
Escala global P99.5: 0.02826625


In [7]:
images_batch, labels_batch = next(iter(train_loader))
print('Images:', images_batch.shape)
print('Labels:', labels_batch.shape)
print('Min:', images_batch.min().item())
print('Max:', images_batch.max().item())
print('Mean:', images_batch.mean().item())
print('Std:', images_batch.std().item())

Images: torch.Size([16, 3, 64, 64])
Labels: torch.Size([16])
Min: -1.0
Max: 1.0
Mean: -0.996368944644928
Std: 0.048466820269823074


## VAE and Latent Preparation

The convolutional VAE compresses each jet into a latent tensor with shape $[4,16,16]$. The posterior means are standardized with statistics computed only from the training set, producing the latent dataset used by the patchwise quantum denoiser.

In [10]:
class ConvBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1,
    ):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                stride=stride,
                padding=1,
            ),
            nn.GroupNorm(
                8,
                out_channels,
            ),
            nn.SiLU(),
        )

    def forward(self, x):
        return self.block(x)

In [11]:
class SimpleConvVAE16Occupancy(nn.Module):

    def __init__(
        self,
        in_channels=3,
        latent_channels=4,
        base_channels=64,
    ):
        super().__init__()

        self.latent_channels = latent_channels

        self.encoder = nn.Sequential(

            ConvBlock(
                in_channels,
                base_channels,
                stride=1,
            ),

            ConvBlock(
                base_channels,
                base_channels * 2,
                stride=2,
            ),

            ConvBlock(
                base_channels * 2,
                base_channels * 4,
                stride=2,
            ),
        )

        self.to_mu = nn.Conv2d(
            base_channels * 4,
            latent_channels,
            kernel_size=3,
            padding=1,
        )

        self.to_log_var = nn.Conv2d(
            base_channels * 4,
            latent_channels,
            kernel_size=3,
            padding=1,
        )

        self.dec_in = ConvBlock(
            latent_channels,
            base_channels * 4,
            stride=1,
        )

        self.up1 = nn.Sequential(

            nn.ConvTranspose2d(
                base_channels * 4,
                base_channels * 2,
                kernel_size=4,
                stride=2,
                padding=1,
            ),

            nn.GroupNorm(
                8,
                base_channels * 2,
            ),

            nn.SiLU(),
        )

        self.up2 = nn.Sequential(

            nn.ConvTranspose2d(
                base_channels * 2,
                base_channels,
                kernel_size=4,
                stride=2,
                padding=1,
            ),

            nn.GroupNorm(
                8,
                base_channels,
            ),

            nn.SiLU(),
        )

        self.mask_head = nn.Conv2d(
            base_channels,
            in_channels,
            kernel_size=3,
            padding=1,
        )

        self.intensity_head = nn.Conv2d(
            base_channels,
            in_channels,
            kernel_size=3,
            padding=1,
        )

        nn.init.zeros_(
            self.mask_head.weight
        )

        nn.init.zeros_(
            self.intensity_head.weight
        )

        mask_prior = 0.01

        mask_bias = np.log(
            mask_prior
            / (1.0 - mask_prior)
        )

        nn.init.constant_(
            self.mask_head.bias,
            float(mask_bias),
        )

        nn.init.constant_(
            self.intensity_head.bias,
            -2.0,
        )

    def encode(self, x):

        h = self.encoder(x)

        mu = self.to_mu(h)

        log_var = self.to_log_var(h)

        log_var = torch.clamp(
            log_var,
            min=-10.0,
            max=8.0,
        )

        return mu, log_var

    @staticmethod
    def reparameterize(
        mu,
        log_var,
    ):

        std = torch.exp(
            0.5 * log_var
        )

        noise = torch.randn_like(
            std
        )

        return (
            mu
            + noise * std
        )

    def decode_components(
        self,
        z,
    ):

        h = self.dec_in(z)

        h = self.up1(h)

        h = self.up2(h)

        mask_logits = (
            self.mask_head(h)
        )

        mask_probability = torch.sigmoid(
            mask_logits
        )

        intensity_logits = (
            self.intensity_head(h)
        )

        intensity_01 = torch.sigmoid(
            intensity_logits
        )

        reconstruction_01 = (
            mask_probability
            * intensity_01
        )

        reconstruction = (
            2.0 * reconstruction_01
            - 1.0
        )

        return (
            reconstruction,
            mask_logits,
            mask_probability,
            intensity_01,
        )

    def decode(
        self,
        z,
    ):

        (
            reconstruction,
            _,
            _,
            _,
        ) = self.decode_components(z)

        return reconstruction

    def decode_hard(
        self,
        z,
    ):

        (
            _,
            _,
            mask_probability,
            intensity_01,
        ) = self.decode_components(z)

        channel_thresholds = (
            get_occupancy_thresholds(
                mask_probability
            )
        )

        hard_mask = (
            mask_probability
            > channel_thresholds
        ).to(
            intensity_01.dtype
        )

        reconstruction_01 = (
            hard_mask
            * intensity_01
        )

        return (
            2.0 * reconstruction_01
            - 1.0
        )

    def forward(
        self,
        x,
        deterministic=False,
        return_components=False,
    ):

        mu, log_var = self.encode(x)

        if deterministic:

            z = mu

        else:

            z = self.reparameterize(
                mu,
                log_var,
            )

        (
            reconstruction,
            mask_logits,
            mask_probability,
            intensity_01,
        ) = self.decode_components(z)

        if return_components:

            return (
                reconstruction,
                mu,
                log_var,
                z,
                mask_logits,
                mask_probability,
                intensity_01,
            )

        return (
            reconstruction,
            mu,
            log_var,
            z,
        )

In [13]:
def masked_mean(values, mask, eps=1e-08):
    return (values * mask).sum() / (mask.sum() + eps)


def jet_vae_loss(
    reconstruction, target, mu, log_var, mask_logits, mask_probability, intensity_01, beta, active_thr_01=0.0001,
    active_weight=5.0, background_weight=1.0, energy_weight=1.0, background_energy_weight=0.01, mask_weight=0.1,
    occupancy_weight=2.0, false_positive_weight=0.5, peak_weight=1.0, occupancy_threshold=0.8,
    gate_temperature=0.05, topk_peak_pixels=8,
):
    target_01 = ((target + 1.0) / 2.0).clamp(0.0, 1.0)
    active_mask = (target_01 > active_thr_01).to(target_01.dtype)
    background_mask = 1.0 - active_mask
    intensity_absolute_error = torch.abs(intensity_01 - target_01)
    intensity_pixel_weight = 1.0 + 4.0 * target_01.square()
    active_intensity_loss = masked_mean(intensity_pixel_weight * intensity_absolute_error, active_mask)
    soft_gate = torch.sigmoid((mask_probability - occupancy_threshold) / gate_temperature)
    predicted_01 = soft_gate * intensity_01
    background_l1 = masked_mean(predicted_01, background_mask)
    reconstruction_loss = active_weight * active_intensity_loss + background_weight * background_l1
    predicted_active_fraction = soft_gate.mean(dim=(-2, -1))
    target_active_fraction = active_mask.mean(dim=(-2, -1))
    occupancy_loss = F.l1_loss(predicted_active_fraction, target_active_fraction)
    false_positive_loss = masked_mean(soft_gate, background_mask)
    predicted_energy = predicted_01.sum(dim=(-2, -1))
    target_energy = target_01.sum(dim=(-2, -1))
    energy_loss = F.smooth_l1_loss(torch.log1p(predicted_energy), torch.log1p(target_energy))
    false_background_energy = (predicted_01 * background_mask).sum(dim=(-2, -1))
    background_energy_loss = torch.log1p(false_background_energy).mean()
    positive_count = active_mask.sum(dim=(0, 2, 3))
    negative_count = background_mask.sum(dim=(0, 2, 3))
    positive_weight = torch.sqrt(negative_count / positive_count.clamp_min(1.0)).clamp(min=1.0, max=6.0)
    positive_weight = positive_weight.view(1, -1, 1, 1).detach()
    mask_loss = F.binary_cross_entropy_with_logits(mask_logits, active_mask, pos_weight=positive_weight)
    num_pixels = target_01.shape[-2] * target_01.shape[-1]
    k = min(topk_peak_pixels, num_pixels)
    predicted_topk = predicted_01.flatten(start_dim=2).topk(k, dim=-1).values.mean(dim=-1)
    target_topk = target_01.flatten(start_dim=2).topk(k, dim=-1).values.mean(dim=-1)
    predicted_maximum = predicted_01.amax(dim=(-2, -1))
    target_maximum = target_01.amax(dim=(-2, -1))
    topk_loss = F.l1_loss(predicted_topk, target_topk)
    maximum_loss = F.l1_loss(predicted_maximum, target_maximum)
    peak_loss = 0.5 * topk_loss + 0.5 * maximum_loss
    kl_loss = 0.5 * (mu.square() + log_var.exp() - 1.0 - log_var).mean()
    total_loss = (
        reconstruction_loss
        + energy_weight * energy_loss
        + background_energy_weight * background_energy_loss
        + mask_weight * mask_loss
        + occupancy_weight * occupancy_loss
        + false_positive_weight * false_positive_loss
        + peak_weight * peak_loss
        + beta * kl_loss
    )
    components = {
        'loss': total_loss,
        'reconstruction': reconstruction_loss,
        'active_l1': active_intensity_loss,
        'background_l1': background_l1,
        'energy': energy_loss,
        'background_energy': background_energy_loss,
        'mask_bce': mask_loss,
        'occupancy': occupancy_loss,
        'false_positive': false_positive_loss,
        'peak': peak_loss,
        'maximum': maximum_loss,
        'kl': kl_loss,
    }

    return (total_loss, components)

In [191]:
def accumulate_metrics(metric_sums, components, batch_size):
    for key, value in components.items():
        metric_sums[key] = metric_sums.get(key, 0.0) + value.detach().item() * batch_size


def average_metrics(metric_sums, n_examples):
    return {key: value / max(n_examples, 1) for key, value in metric_sums.items()}


def beta_for_epoch(epoch, beta_max, kl_start_epoch, kl_warmup_epochs):
    if epoch < kl_start_epoch:
        return 0.0

    progress = min(1.0, (epoch - kl_start_epoch + 1) / max(kl_warmup_epochs, 1))

    return beta_max * progress


@torch.no_grad()
def evaluate_vae_epoch(model, data_loader, beta, device):
    model.eval()
    totals = {}
    n_examples = 0

    for images, _ in data_loader:
        images = images.to(device)
        reconstruction, mu, log_var, _, mask_logits, mask_probability, intensity_01 = model(images, deterministic=True, return_components=True)
        _, components = jet_vae_loss(
            reconstruction=reconstruction, target=images, mu=mu, log_var=log_var, mask_logits=mask_logits,
            mask_probability=mask_probability, intensity_01=intensity_01, beta=beta,
            active_thr_01=ACTIVE_THRESHOLD_01, active_weight=ACTIVE_WEIGHT, background_weight=BACKGROUND_WEIGHT,
            energy_weight=ENERGY_WEIGHT, background_energy_weight=BACKGROUND_ENERGY_WEIGHT,
            mask_weight=MASK_WEIGHT, occupancy_weight=OCCUPANCY_WEIGHT,
            false_positive_weight=FALSE_POSITIVE_WEIGHT, peak_weight=PEAK_WEIGHT,
            occupancy_threshold=OCCUPANCY_THRESHOLD, gate_temperature=GATE_TEMPERATURE,
            topk_peak_pixels=TOPK_PEAK_PIXELS,
        )
        batch_size = images.size(0)

        accumulate_metrics(totals, components, batch_size)
        n_examples += batch_size

    return average_metrics(totals, n_examples)


def train_jet_vae(
    train_loader, val_loader, device, beta_max, num_epochs=20, learning_rate=0.0002, patience=12,
    model_class=SimpleConvVAE16Occupancy, latent_channels=LATENT_CHANNELS,
):
    model = model_class(in_channels=3, latent_channels=latent_channels, base_channels=BASE_CHANNELS).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.0001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4, min_lr=1e-06)
    history = {'train': [], 'val': [], 'beta': [], 'lr': []}
    best_validation_loss = float('inf')
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(num_epochs):
        beta = beta_for_epoch(epoch=epoch, beta_max=beta_max, kl_start_epoch=KL_START_EPOCH, kl_warmup_epochs=KL_WARMUP_EPOCHS)
        model.train()
        totals = {}
        n_examples = 0

        for images, _ in train_loader:
            images = images.to(device)
            optimizer.zero_grad(set_to_none=True)
            reconstruction, mu, log_var, _, mask_logits, mask_probability, intensity_01 = model(images, deterministic=False, return_components=True)
            loss, components = jet_vae_loss(
                reconstruction=reconstruction, target=images, mu=mu, log_var=log_var, mask_logits=mask_logits,
                mask_probability=mask_probability, intensity_01=intensity_01, beta=beta,
                active_thr_01=ACTIVE_THRESHOLD_01, active_weight=ACTIVE_WEIGHT,
                background_weight=BACKGROUND_WEIGHT, energy_weight=ENERGY_WEIGHT,
                background_energy_weight=BACKGROUND_ENERGY_WEIGHT, mask_weight=MASK_WEIGHT,
                occupancy_weight=OCCUPANCY_WEIGHT, false_positive_weight=FALSE_POSITIVE_WEIGHT,
                peak_weight=PEAK_WEIGHT, occupancy_threshold=OCCUPANCY_THRESHOLD,
                gate_temperature=GATE_TEMPERATURE, topk_peak_pixels=TOPK_PEAK_PIXELS,
            )

            if not torch.isfinite(loss):
                raise RuntimeError(f'Loss no finita en época {epoch + 1}: {loss.item()}')

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            batch_size = images.size(0)
            accumulate_metrics(totals, components, batch_size)
            n_examples += batch_size

        train_metrics = average_metrics(totals, n_examples)
        validation_metrics = evaluate_vae_epoch(model=model, data_loader=val_loader, beta=beta, device=device)
        scheduler.step(validation_metrics['loss'])
        current_lr = optimizer.param_groups[0]['lr']
        history['train'].append(train_metrics)
        history['val'].append(validation_metrics)
        history['beta'].append(beta)
        history['lr'].append(current_lr)
        print(
            f"Epoch {epoch + 1:03d}/{num_epochs} | "
            f"beta={beta:.2e} | "
            f"lr={current_lr:.2e} | "
            f"train={train_metrics['loss']:.6f} | "
            f"val={validation_metrics['loss']:.6f} | "
            f"active={validation_metrics['active_l1']:.6f} | "
            f"bg={validation_metrics['background_l1']:.6f} | "
            f"energy={validation_metrics['energy']:.6f} | "
            f"occupancy={validation_metrics['occupancy']:.6f} | "
            f"false_pos={validation_metrics['false_positive']:.6f} | "
            f"peak={validation_metrics['peak']:.6f} | "
            f"max={validation_metrics['maximum']:.6f} | "
            f"mask={validation_metrics['mask_bce']:.6f} | "
            f"KL={validation_metrics['kl']:.6f}"
        )

        if validation_metrics['loss'] < best_validation_loss - 1e-06:
            best_validation_loss = validation_metrics['loss']
            best_state = deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f'Early stopping en época {epoch + 1}. Mejor val loss: {best_validation_loss:.6f}')

            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return (model.to(device), history)

In [15]:
seed_everything(SEED)
vae_model, vae_history = train_jet_vae(
    train_loader=train_loader, val_loader=val_loader, device=device, beta_max=BETA_MAX, num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE, patience=20,
)

Epoch 001/40 | beta=0.00e+00 | lr=2.00e-04 | train=0.847529 | val=0.450614 | active=0.040202 | bg=0.000809 | energy=0.033659 | occupancy=0.015164 | false_pos=0.027541 | peak=0.105452 | max=0.132557 | mask=0.523436 | KL=3.097813
Epoch 002/40 | beta=0.00e+00 | lr=2.00e-04 | train=0.398274 | val=0.364601 | active=0.032800 | bg=0.000726 | energy=0.018387 | occupancy=0.012255 | false_pos=0.026753 | peak=0.084950 | max=0.109561 | mask=0.464102 | KL=4.136959
Epoch 003/40 | beta=0.00e+00 | lr=2.00e-04 | train=0.331736 | val=0.309191 | active=0.029368 | bg=0.000541 | energy=0.012040 | occupancy=0.010082 | false_pos=0.022470 | peak=0.069480 | max=0.094733 | mask=0.388975 | KL=4.778061
Epoch 004/40 | beta=0.00e+00 | lr=2.00e-04 | train=0.286885 | val=0.281717 | active=0.026907 | bg=0.000454 | energy=0.011993 | occupancy=0.011825 | false_pos=0.017964 | peak=0.058308 | max=0.078392 | mask=0.350906 | KL=5.020541
Epoch 005/40 | beta=0.00e+00 | lr=2.00e-04 | train=0.263097 | val=0.267519 | active=0.02

In [33]:
LDM_BATCH_SIZE = 64
LDM_ENCODING_BATCH_SIZE = 64
vae_model.eval()

for parameter in vae_model.parameters():
    parameter.requires_grad_(False)

print('VAE congelado para el entrenamiento del LDM.')

VAE congelado para el entrenamiento del LDM.


In [34]:
@torch.no_grad()
def encode_dataset_to_mu(model, dataset, device, batch_size=64):
    model.eval()
    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    mu_batches = []
    label_batches = []

    for images, labels in tqdm(data_loader, desc='Codificando latentes'):
        images = images.to(device, non_blocking=True)
        mu, _ = model.encode(images)
        mu_batches.append(mu.detach().cpu())
        label_batches.append(labels.detach().cpu())

    mu_all = torch.cat(mu_batches, dim=0)
    labels_all = torch.cat(label_batches, dim=0)

    return (mu_all, labels_all)

In [35]:
ldm_train_mu, ldm_train_labels = encode_dataset_to_mu(model=vae_model, dataset=train_dataset, device=device, batch_size=LDM_ENCODING_BATCH_SIZE)
ldm_val_mu, ldm_val_labels = encode_dataset_to_mu(model=vae_model, dataset=val_dataset, device=device, batch_size=LDM_ENCODING_BATCH_SIZE)
ldm_test_mu, ldm_test_labels = encode_dataset_to_mu(model=vae_model, dataset=test_dataset, device=device, batch_size=LDM_ENCODING_BATCH_SIZE)

print('Train:', ldm_train_mu.shape)
print('Validation:', ldm_val_mu.shape)
print('Test:', ldm_test_mu.shape)

Codificando latentes: 100%|██████████| 40/40 [00:00<00:00, 50.00it/s]

Train: torch.Size([10000, 4, 16, 16])
Validation: torch.Size([2500, 4, 16, 16])
Test: torch.Size([2500, 4, 16, 16])


In [36]:
latent_mean_train = ldm_train_mu.mean(dim=(0, 2, 3), keepdim=True)
latent_std_train = ldm_train_mu.std(dim=(0, 2, 3), keepdim=True, unbiased=False).clamp_min(1e-06)
print('Media latente de entrenamiento:', latent_mean_train.flatten().numpy())
print('Std latente de entrenamiento:', latent_std_train.flatten().numpy())

Media latente de entrenamiento: [-1.065183   -0.64281386  0.8292197  -0.60205144]
Std latente de entrenamiento: [1.0347776  1.0140498  0.78007245 0.85224116]


In [37]:
def standardize_latents(latent_tensor, latent_mean, latent_std):
    return (latent_tensor - latent_mean) / latent_std.clamp_min(1e-06)

In [38]:
ldm_train_latents = standardize_latents(ldm_train_mu, latent_mean_train, latent_std_train)
ldm_val_latents = standardize_latents(ldm_val_mu, latent_mean_train, latent_std_train)
ldm_test_latents = standardize_latents(ldm_test_mu, latent_mean_train, latent_std_train)

In [39]:
del ldm_train_mu
del ldm_val_mu
del ldm_test_mu

In [40]:
def summarize_latent_tensor(latent_tensor, name):
    channel_mean = latent_tensor.mean(dim=(0, 2, 3))
    channel_std = latent_tensor.std(dim=(0, 2, 3), unbiased=False)
    absolute_latents = latent_tensor.abs()

    print(f'\n{name}')
    print('Shape:', tuple(latent_tensor.shape))
    print('Media por canal:', channel_mean.numpy())
    print('Std por canal:', channel_std.numpy())
    print('Media global:', latent_tensor.mean().item())
    print('Std global:', latent_tensor.std(unbiased=False).item())
    print('Mínimo:', latent_tensor.min().item())
    print('Máximo:', latent_tensor.max().item())
    print('Fracción |z| > 4:', (absolute_latents > 4.0).float().mean().item())
    print('Fracción |z| > 6:', (absolute_latents > 6.0).float().mean().item())

In [41]:
summarize_latent_tensor(ldm_train_latents, 'LATENTES DE ENTRENAMIENTO')
summarize_latent_tensor(ldm_val_latents, 'LATENTES DE VALIDACIÓN')
summarize_latent_tensor(ldm_test_latents, 'LATENTES DE TEST')


LATENTES DE ENTRENAMIENTO
Shape: (10000, 4, 16, 16)
Media por canal: [ 2.3272262e-06  1.4475598e-06 -1.9278228e-07  1.7276809e-06]
Std por canal: [1.         0.99999994 1.         0.99999994]
Media global: 1.3257980526759638e-06
Std global: 1.0
Mínimo: -20.84419822692871
Máximo: 21.923025131225586
Fracción |z| > 4: 0.00910634733736515
Fracción |z| > 6: 0.003725390648469329

LATENTES DE VALIDACIÓN
Shape: (2500, 4, 16, 16)
Media por canal: [ 0.00084296 -0.00137886 -0.00233332 -0.00041057]
Std por canal: [1.0002276  1.0007932  0.9940807  0.99733704]
Media global: -0.0008199462899938226
Std global: 0.9981139302253723
Mínimo: -18.906015396118164
Máximo: 19.29391098022461
Fracción |z| > 4: 0.009114062413573265
Fracción |z| > 6: 0.0036746093537658453

LATENTES DE TEST
Shape: (2500, 4, 16, 16)
Media por canal: [ 0.00182807 -0.0012871  -0.00110344 -0.00018797]
Std por canal: [1.003436   0.9966871  0.9964558  0.99847895]
Media global: -0.00018760976672638208
Std global: 0.9987691640853882
Mínim

In [42]:
ldm_train_dataset = TensorDataset(ldm_train_latents, ldm_train_labels)
ldm_val_dataset = TensorDataset(ldm_val_latents, ldm_val_labels)
ldm_test_dataset = TensorDataset(ldm_test_latents, ldm_test_labels)

In [43]:
ldm_train_generator = torch.Generator().manual_seed(SEED)
ldm_train_loader = DataLoader(ldm_train_dataset, batch_size=LDM_BATCH_SIZE, shuffle=True, num_workers=0, generator=ldm_train_generator)
ldm_val_loader = DataLoader(ldm_val_dataset, batch_size=LDM_BATCH_SIZE, shuffle=False, num_workers=0)
ldm_test_loader = DataLoader(ldm_test_dataset, batch_size=LDM_BATCH_SIZE, shuffle=False, num_workers=0)

## Diffusion Process and Training Objective

The linear noise scheduler defines the forward DDPM process. The model is trained with `v`-prediction loss, which compares the predicted velocity with the analytical target and includes an additional reconstruction penalty for the clean latent representation.

In [46]:
class LinearNoiseScheduler:
    def __init__(self, num_timesteps, beta_s, beta_e, device):
        self.num_timesteps = num_timesteps
        self.beta_s = beta_s
        self.beta_e = beta_e
        self.device = device

        self.betas = torch.linspace(beta_s, beta_e, num_timesteps, device=device)
        self.alphas = 1.0 - self.betas
        self.alpha_cum_prod = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alpha_cum_prod = torch.sqrt(self.alpha_cum_prod)
        self.sqrt_one_minus_alpha_cum_prod = torch.sqrt(1.0 - self.alpha_cum_prod)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / self.alphas)
        alpha_cum_prod_prev = torch.cat([torch.tensor([1.0], device=device), self.alpha_cum_prod[:-1]], dim=0)
        self.posterior_variance = (self.betas * (1.0 - alpha_cum_prod_prev) / (1.0 - self.alpha_cum_prod))
        self.posterior_mean_coef1 = (self.betas * torch.sqrt(alpha_cum_prod_prev) / (1.0 - self.alpha_cum_prod))
        self.posterior_mean_coef2 = (torch.sqrt(self.alphas) * (1.0 - alpha_cum_prod_prev) / (1.0 - self.alpha_cum_prod))

    def sample_timesteps(self, batch_size):
        return torch.randint(0, self.num_timesteps, (batch_size,), device=self.device)

    def _extract(self, arr, t, x_shape):
        out = arr.gather(0, t)

        return out.view(t.shape[0], *([1] * (len(x_shape) - 1)))

    def add_noise(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)

        sqrt_alpha_cum_prod_t = self._extract(self.sqrt_alpha_cum_prod, t, x0.shape)
        sqrt_one_minus_alpha_cum_prod_t = self._extract(self.sqrt_one_minus_alpha_cum_prod, t, x0.shape)
        xt = sqrt_alpha_cum_prod_t * x0 + sqrt_one_minus_alpha_cum_prod_t * noise

        return xt, noise

    @torch.no_grad()
    def sample_prev_timestep_v(self, xt, v_pred, t, clip_x0=True):

        if isinstance(t, torch.Tensor):
            t_int = int(t.item())
        else:
            t_int = int(t)

        sqrt_alpha_bar_t = self.sqrt_alpha_cum_prod[t_int]
        sqrt_one_minus_alpha_bar_t = self.sqrt_one_minus_alpha_cum_prod[t_int]
        x0_pred = sqrt_alpha_bar_t * xt - sqrt_one_minus_alpha_bar_t * v_pred

        if clip_x0:
            x0_pred = torch.clamp(x0_pred, -1.0, 1.0)

        if t_int == 0:
            return x0_pred, x0_pred

        posterior_mean = (self.posterior_mean_coef1[t_int] * x0_pred + self.posterior_mean_coef2[t_int] * xt)
        posterior_var = self.posterior_variance[t_int]
        noise = torch.randn_like(xt)
        x_prev = posterior_mean + torch.sqrt(posterior_var) * noise

        return x_prev, x0_pred

In [49]:
LDM_NUM_TIMESTEPS = 1000
LDM_BETA_START = 0.0001
LDM_BETA_END = 0.02
LDM_NUM_EPOCHS = 100
LDM_LEARNING_RATE = 0.0001
LDM_GRAD_CLIP = 1.0
NUM_CLASSES = 2

In [53]:
def compute_v_target(z0, noise, timesteps, scheduler):
    sqrt_alpha_bar = scheduler._extract(scheduler.sqrt_alpha_cum_prod, timesteps, z0.shape)
    sqrt_one_minus_alpha_bar = scheduler._extract(scheduler.sqrt_one_minus_alpha_cum_prod, timesteps, z0.shape)
    v_target = sqrt_alpha_bar * noise - sqrt_one_minus_alpha_bar * z0

    return v_target

In [55]:
def latent_v_loss(z0, z_t, v_prediction, v_target, timesteps, scheduler, tail_loss_weight=0.15):
    v_loss = F.mse_loss(v_prediction, v_target)
    sqrt_alpha_bar = scheduler._extract(scheduler.sqrt_alpha_cum_prod, timesteps, z0.shape)
    sqrt_one_minus_alpha_bar = scheduler._extract(scheduler.sqrt_one_minus_alpha_cum_prod, timesteps, z0.shape)
    z0_prediction = sqrt_alpha_bar * z_t - sqrt_one_minus_alpha_bar * v_prediction
    tail_weight = 1.0 + 2.0 * ((z0.abs() - 2.0) / 4.0).clamp(min=0.0, max=1.0)
    reconstruction_error = F.smooth_l1_loss(z0_prediction, z0, reduction='none')
    tail_loss = (tail_weight * reconstruction_error).mean()
    total_loss = v_loss + tail_loss_weight * tail_loss

    return (total_loss, {'loss': total_loss, 'v_loss': v_loss, 'tail_loss': tail_loss})

## Pure Patchwise Quantum Denoiser

The latent tensor is divided into 64 spatial patches of shape $[4,2,2]$. Each patch therefore contains 16 values, which are loaded into the same eight-qubit PQC in two stages. The circuit also receives the timestep, class label, and patch coordinates and returns 16 observables. There is no U-Net, classical denoising backbone, or bypass around the quantum circuit.

In [200]:
import pennylane as qml
QPD_DEVICE = torch.device('cpu')
QPD_NUM_QUBITS = 8
QPD_PATCH_SIZE = 2
QPD_LATENT_CHANNELS = 4
QPD_PATCH_DIM = QPD_LATENT_CHANNELS * QPD_PATCH_SIZE * QPD_PATCH_SIZE
QPD_GRID_SIZE = 16 // QPD_PATCH_SIZE
QPD_NUM_PATCHES = QPD_GRID_SIZE * QPD_GRID_SIZE
QPD_LAYERS_PER_STAGE = 2

print('Qubits:', QPD_NUM_QUBITS)
print('Values per patch:', QPD_PATCH_DIM)
print('Patches per latent:', QPD_NUM_PATCHES)

Qubits: 8
Values per patch: 16
Patches per latent: 64


In [201]:
def latent_to_qpd_patches(
    latent,
):

    batch_size, channels, height, width = (
        latent.shape
    )

    if channels != 4:
        raise ValueError(
            f"Expected 4 channels, got {channels}."
        )

    if height != 16 or width != 16:
        raise ValueError(
            "Expected spatial shape 16x16."
        )

    patches = latent.reshape(
        batch_size,
        channels,
        QPD_GRID_SIZE,
        QPD_PATCH_SIZE,
        QPD_GRID_SIZE,
        QPD_PATCH_SIZE,
    )

    patches = patches.permute(
        0,
        2,
        4,
        1,
        3,
        5,
    )

    patches = patches.reshape(
        batch_size,
        QPD_NUM_PATCHES,
        QPD_PATCH_DIM,
    )

    return patches


def qpd_patches_to_latent(
    patches,
):

    batch_size = patches.shape[0]

    latent = patches.reshape(
        batch_size,
        QPD_GRID_SIZE,
        QPD_GRID_SIZE,
        QPD_LATENT_CHANNELS,
        QPD_PATCH_SIZE,
        QPD_PATCH_SIZE,
    )

    latent = latent.permute(
        0,
        3,
        1,
        4,
        2,
        5,
    )

    latent = latent.reshape(
        batch_size,
        QPD_LATENT_CHANNELS,
        16,
        16,
    )

    return latent

In [202]:
qpd_test_latent = ldm_train_latents[:2].float()
qpd_test_patches = latent_to_qpd_patches(qpd_test_latent)
qpd_test_reconstruction = qpd_patches_to_latent(qpd_test_patches)

print('Latent:', qpd_test_latent.shape)
print('Patches:', qpd_test_patches.shape)
print('Reconstructed:', qpd_test_reconstruction.shape)
print('Maximum reconstruction error:', (qpd_test_latent - qpd_test_reconstruction).abs().max().item())

Latent: torch.Size([2, 4, 16, 16])
Patches: torch.Size([2, 64, 16])
Reconstructed: torch.Size([2, 4, 16, 16])
Maximum reconstruction error: 0.0


In [203]:
qpd_quantum_device = qml.device('default.qubit', wires=QPD_NUM_QUBITS, shots=None)


def qpd_variational_block(
    weights,
):

    for qubit in range(
        QPD_NUM_QUBITS
    ):

        qml.RX(
            weights[
                qubit,
                0,
            ],
            wires=qubit,
        )

        qml.RY(
            weights[
                qubit,
                1,
            ],
            wires=qubit,
        )

        qml.RZ(
            weights[
                qubit,
                2,
            ],
            wires=qubit,
        )

    for qubit in range(
        QPD_NUM_QUBITS - 1
    ):

        qml.CNOT(
            wires=[
                qubit,
                qubit + 1,
            ]
        )

    qml.CNOT(
        wires=[
            QPD_NUM_QUBITS - 1,
            0,
        ]
    )


def qpd_conditioning_layer(
    timestep_signal,
    class_signal,
    x_position,
    y_position,
):

    qml.RZ(
        math.pi * timestep_signal,
        wires=0,
    )

    qml.RZ(
        math.pi * timestep_signal,
        wires=1,
    )

    qml.RY(
        math.pi * class_signal,
        wires=2,
    )

    qml.RY(
        math.pi * class_signal,
        wires=3,
    )

    qml.RX(
        math.pi * x_position,
        wires=4,
    )

    qml.RX(
        math.pi * x_position,
        wires=5,
    )

    qml.RY(
        math.pi * y_position,
        wires=6,
    )

    qml.RY(
        math.pi * y_position,
        wires=7,
    )


@qml.qnode(
    qpd_quantum_device,
    interface="torch",
    diff_method="backprop",
)
def qpd_quantum_circuit(
    inputs,
    weights,
):

    first_half = inputs[..., 0:8]

    second_half = inputs[..., 8:16]

    timestep_signal = inputs[..., 16]

    class_signal = inputs[..., 17]

    x_position = inputs[..., 18]

    y_position = inputs[..., 19]

    for qubit in range(
        QPD_NUM_QUBITS
    ):

        angle = (
            2.0
            * torch.atan(
                first_half[..., qubit]
            )
        )

        qml.RY(
            angle,
            wires=qubit,
        )

    qpd_conditioning_layer(
        timestep_signal,
        class_signal,
        x_position,
        y_position,
    )

    for layer in range(
        QPD_LAYERS_PER_STAGE
    ):

        qpd_variational_block(
            weights[
                0,
                layer,
            ]
        )

    for qubit in range(
        QPD_NUM_QUBITS
    ):

        angle = (
            2.0
            * torch.atan(
                second_half[..., qubit]
            )
        )

        qml.RY(
            angle,
            wires=qubit,
        )

    qpd_conditioning_layer(
        timestep_signal,
        class_signal,
        x_position,
        y_position,
    )

    for layer in range(
        QPD_LAYERS_PER_STAGE
    ):

        qpd_variational_block(
            weights[
                1,
                layer,
            ]
        )

    local_outputs = [

        qml.expval(
            qml.PauliZ(qubit)
        )

        for qubit in range(
            QPD_NUM_QUBITS
        )
    ]

    correlation_outputs = [

        qml.expval(
            qml.PauliZ(qubit)
            @
            qml.PauliZ(
                (
                    qubit + 1
                )
                % QPD_NUM_QUBITS
            )
        )

        for qubit in range(
            QPD_NUM_QUBITS
        )
    ]

    return (
        local_outputs
        + correlation_outputs
    )

In [204]:
class PatchwiseQuantumDenoiser(
    nn.Module
):

    def __init__(
        self,
        num_timesteps=1000,
    ):
        super().__init__()

        self.num_timesteps = (
            num_timesteps
        )

        self.quantum_layer = (
            qml.qnn.TorchLayer(
                qpd_quantum_circuit,
                weight_shapes={
                    "weights": (
                        2,
                        QPD_LAYERS_PER_STAGE,
                        QPD_NUM_QUBITS,
                        3,
                    )
                },
            )
        )

        for parameter in (
            self.quantum_layer.parameters()
        ):

            nn.init.normal_(
                parameter,
                mean=0.0,
                std=0.05,
            )

        self.output_scale = nn.Parameter(
            torch.full(
                (QPD_PATCH_DIM,),
                0.10,
            )
        )

        self.output_bias = nn.Parameter(
            torch.zeros(
                QPD_PATCH_DIM
            )
        )

        coordinate_axis = torch.linspace(
            0.0,
            1.0,
            QPD_GRID_SIZE,
        )

        y_grid, x_grid = torch.meshgrid(
            coordinate_axis,
            coordinate_axis,
            indexing="ij",
        )

        patch_positions = torch.stack(
            [
                x_grid.reshape(-1),
                y_grid.reshape(-1),
            ],
            dim=1,
        )

        self.register_buffer(
            "patch_positions",
            patch_positions,
        )

    def forward(
        self,
        noisy_latent,
        timesteps,
        labels,
    ):

        batch_size = (
            noisy_latent.shape[0]
        )

        patches = (
            latent_to_qpd_patches(
                noisy_latent
            )
        )

        timestep_signal = (
            timesteps.float()
            / float(
                self.num_timesteps - 1
            )
        )

        timestep_signal = (
            timestep_signal[
                :,
                None,
                None,
            ]
            .expand(
                -1,
                QPD_NUM_PATCHES,
                1,
            )
        )

        class_signal = (
            labels.float()[
                :,
                None,
                None,
            ]
            .expand(
                -1,
                QPD_NUM_PATCHES,
                1,
            )
        )

        patch_positions = (
            self.patch_positions[
                None,
                :,
                :,
            ]
            .expand(
                batch_size,
                -1,
                -1,
            )
        )

        quantum_inputs = torch.cat(
            [
                patches,
                timestep_signal,
                class_signal,
                patch_positions,
            ],
            dim=-1,
        )

        quantum_inputs = (
            quantum_inputs.reshape(
                batch_size
                * QPD_NUM_PATCHES,
                20,
            )
        )

        quantum_outputs = (
            self.quantum_layer(
                quantum_inputs
            )
            .float()
        )

        quantum_outputs = (
            quantum_outputs.reshape(
                batch_size,
                QPD_NUM_PATCHES,
                QPD_PATCH_DIM,
            )
        )

        predicted_patches = (
            quantum_outputs
            * self.output_scale[
                None,
                None,
                :,
            ]
            + self.output_bias[
                None,
                None,
                :,
            ]
        )

        predicted_v = (
            qpd_patches_to_latent(
                predicted_patches
            )
        )

        return predicted_v

In [205]:
seed_everything(SEED + 8000)
qpd_model = PatchwiseQuantumDenoiser(num_timesteps=LDM_NUM_TIMESTEPS).to(QPD_DEVICE)
qpd_total_parameters = sum((p.numel() for p in qpd_model.parameters() if p.requires_grad))
qpd_quantum_parameters = sum((p.numel() for p in qpd_model.quantum_layer.parameters() if p.requires_grad))

print('Total trainable parameters:', qpd_total_parameters)
print('Quantum parameters:', qpd_quantum_parameters)
print('Classical scale/bias:', qpd_total_parameters - qpd_quantum_parameters)

Total trainable parameters: 128
Quantum parameters: 96
Classical scale/bias: 32


In [206]:
qpd_scheduler = LinearNoiseScheduler(num_timesteps=LDM_NUM_TIMESTEPS, beta_s=LDM_BETA_START, beta_e=LDM_BETA_END, device=QPD_DEVICE)

### Fixed-Batch Capacity Test

The experiment uses the same four jets, timesteps, Gaussian noise, and `v` targets throughout training. Its purpose is to determine whether the shared patchwise PQC can outperform the zero-velocity predictor before attempting a larger pilot study.

In [207]:
qpd_class_0_indices = torch.where(ldm_train_labels == 0)[0][:2]
qpd_class_1_indices = torch.where(ldm_train_labels == 1)[0][:2]
qpd_fixed_indices = torch.cat([qpd_class_0_indices, qpd_class_1_indices])
qpd_fixed_z0 = ldm_train_latents[qpd_fixed_indices].float().to(QPD_DEVICE)
qpd_fixed_labels = ldm_train_labels[qpd_fixed_indices].long().to(QPD_DEVICE)
qpd_fixed_timesteps = torch.tensor([100, 300, 600, 900], dtype=torch.long, device=QPD_DEVICE)
qpd_noise_generator = torch.Generator(device='cpu').manual_seed(SEED + 8100)
qpd_fixed_noise = torch.randn(qpd_fixed_z0.shape, generator=qpd_noise_generator, dtype=torch.float32).to(QPD_DEVICE)
qpd_fixed_zt, qpd_fixed_noise = qpd_scheduler.add_noise(qpd_fixed_z0, qpd_fixed_timesteps, qpd_fixed_noise)
qpd_fixed_v_target = compute_v_target(z0=qpd_fixed_z0, noise=qpd_fixed_noise, timesteps=qpd_fixed_timesteps, scheduler=qpd_scheduler)

print('z0:', qpd_fixed_z0.shape)
print('zt:', qpd_fixed_zt.shape)
print('v target:', qpd_fixed_v_target.shape)
print('Labels:', qpd_fixed_labels.tolist())

z0: torch.Size([4, 4, 16, 16])
zt: torch.Size([4, 4, 16, 16])
v target: torch.Size([4, 4, 16, 16])
Labels: [0, 0, 1, 1]


In [208]:
with torch.no_grad():
    qpd_zero_prediction = torch.zeros_like(qpd_fixed_v_target)
    qpd_zero_loss, qpd_zero_components = latent_v_loss(
        z0=qpd_fixed_z0, z_t=qpd_fixed_zt, v_prediction=qpd_zero_prediction, v_target=qpd_fixed_v_target,
        timesteps=qpd_fixed_timesteps, scheduler=qpd_scheduler, tail_loss_weight=0.15,
    )

print('Fixed-batch zero-v loss:', qpd_zero_loss.item())

Fixed-batch zero-v loss: 0.8957917094230652


In [209]:
seed_everything(SEED + 8200)
qpd_optimizer = Adam(qpd_model.parameters(), lr=0.003)
QPD_CAPACITY_STEPS = 150
qpd_capacity_history = []
best_qpd_loss = float('inf')

for step in range(1, QPD_CAPACITY_STEPS + 1):
    qpd_model.train()
    qpd_optimizer.zero_grad(set_to_none=True)
    qpd_prediction = qpd_model(qpd_fixed_zt, qpd_fixed_timesteps, qpd_fixed_labels)
    qpd_loss, qpd_components = latent_v_loss(
        z0=qpd_fixed_z0, z_t=qpd_fixed_zt, v_prediction=qpd_prediction, v_target=qpd_fixed_v_target,
        timesteps=qpd_fixed_timesteps, scheduler=qpd_scheduler, tail_loss_weight=0.15,
    )
    qpd_loss.backward()
    torch.nn.utils.clip_grad_norm_(qpd_model.parameters(), max_norm=1.0)
    qpd_optimizer.step()

    current_loss = qpd_loss.detach().item()
    qpd_capacity_history.append(current_loss)
    best_qpd_loss = min(best_qpd_loss, current_loss)

    if step == 1 or step % 25 == 0:
        improvement = 100.0 * (qpd_zero_loss.item() - current_loss) / qpd_zero_loss.item()
        print(f'Step {step:03d}/{QPD_CAPACITY_STEPS} | loss={current_loss:.6f} | improvement vs zero={improvement:+.2f}%')

best_improvement = 100.0 * (qpd_zero_loss.item() - best_qpd_loss) / qpd_zero_loss.item()
print('\nZero-v:', f'{qpd_zero_loss.item():.6f}')
print('Best quantum:', f'{best_qpd_loss:.6f}')
print('Best improvement:', f'{best_improvement:.2f}%')

Step 001/150 | loss=0.895753 | improvement vs zero=+0.00%
Step 025/150 | loss=0.887883 | improvement vs zero=+0.88%
Step 050/150 | loss=0.884430 | improvement vs zero=+1.27%
Step 075/150 | loss=0.880813 | improvement vs zero=+1.67%
Step 100/150 | loss=0.877236 | improvement vs zero=+2.07%
Step 125/150 | loss=0.873533 | improvement vs zero=+2.48%
Step 150/150 | loss=0.869687 | improvement vs zero=+2.91%

Zero-v: 0.895792
Best quantum: 0.869687
Best improvement: 2.91%


## Quantum-Local and Classical-Spatial Denoiser

This variant retains the same shared eight-qubit PQC as the pure patchwise model and adds two minimal depthwise $3\times3$ convolutions. These layers communicate information between neighboring positions without mixing latent channels. The spatial mixer receives only quantum outputs, so there is no direct classical bypass from the noisy latent tensor.

In [210]:
class SpatialPatchwiseQuantumDenoiser(
    PatchwiseQuantumDenoiser
):

    def __init__(
        self,
        num_timesteps=1000,
    ):

        super().__init__(
            num_timesteps=num_timesteps
        )

        self.spatial_dw1 = nn.Conv2d(
            in_channels=QPD_LATENT_CHANNELS,
            out_channels=QPD_LATENT_CHANNELS,
            kernel_size=3,
            padding=1,
            groups=QPD_LATENT_CHANNELS,
            bias=True,
        )

        self.spatial_dw2 = nn.Conv2d(
            in_channels=QPD_LATENT_CHANNELS,
            out_channels=QPD_LATENT_CHANNELS,
            kernel_size=3,
            padding=1,
            groups=QPD_LATENT_CHANNELS,
            bias=True,
        )

        nn.init.normal_(
            self.spatial_dw1.weight,
            mean=0.0,
            std=0.01,
        )

        nn.init.zeros_(
            self.spatial_dw1.bias
        )

        nn.init.normal_(
            self.spatial_dw2.weight,
            mean=0.0,
            std=0.01,
        )

        nn.init.zeros_(
            self.spatial_dw2.bias
        )

    def forward(
        self,
        noisy_latent,
        timesteps,
        labels,
    ):

        quantum_prediction = (
            super().forward(
                noisy_latent,
                timesteps,
                labels,
            )
        )

        spatial_features = (
            self.spatial_dw1(
                quantum_prediction
            )
        )

        spatial_features = F.silu(
            spatial_features
        )

        spatial_delta = (
            self.spatial_dw2(
                spatial_features
            )
        )

        prediction = (
            quantum_prediction
            + spatial_delta
        )

        return prediction

In [211]:
seed_everything(SEED + 8300)
qpd_spatial_model = SpatialPatchwiseQuantumDenoiser(num_timesteps=LDM_NUM_TIMESTEPS).to(QPD_DEVICE)
qpd_spatial_total_parameters = sum((parameter.numel() for parameter in qpd_spatial_model.parameters() if parameter.requires_grad))
qpd_spatial_quantum_parameters = sum((parameter.numel() for parameter in qpd_spatial_model.quantum_layer.parameters() if parameter.requires_grad))
qpd_spatial_mixer_parameters = (
    sum((parameter.numel() for parameter in qpd_spatial_model.spatial_dw1.parameters()))
    + sum((parameter.numel() for parameter in qpd_spatial_model.spatial_dw2.parameters()))
)

print('Total trainable:', qpd_spatial_total_parameters)
print('Quantum:', qpd_spatial_quantum_parameters)
print('Output scale/bias:', 32)
print('Spatial mixer:', qpd_spatial_mixer_parameters)

Total trainable: 208
Quantum: 96
Output scale/bias: 32
Spatial mixer: 80


In [212]:
seed_everything(SEED + 8400)
qpd_spatial_optimizer = Adam(qpd_spatial_model.parameters(), lr=0.003)
QPD_SPATIAL_CAPACITY_STEPS = 150
qpd_spatial_history = []
best_qpd_spatial_loss = float('inf')

for step in range(1, QPD_SPATIAL_CAPACITY_STEPS + 1):
    qpd_spatial_model.train()
    qpd_spatial_optimizer.zero_grad(set_to_none=True)
    qpd_spatial_prediction = qpd_spatial_model(qpd_fixed_zt, qpd_fixed_timesteps, qpd_fixed_labels)
    qpd_spatial_loss, qpd_spatial_components = latent_v_loss(
        z0=qpd_fixed_z0, z_t=qpd_fixed_zt, v_prediction=qpd_spatial_prediction, v_target=qpd_fixed_v_target,
        timesteps=qpd_fixed_timesteps, scheduler=qpd_scheduler, tail_loss_weight=0.15,
    )
    qpd_spatial_loss.backward()
    torch.nn.utils.clip_grad_norm_(qpd_spatial_model.parameters(), max_norm=1.0)
    qpd_spatial_optimizer.step()

    current_loss = qpd_spatial_loss.detach().item()
    qpd_spatial_history.append(current_loss)
    best_qpd_spatial_loss = min(best_qpd_spatial_loss, current_loss)

    if step == 1 or step % 25 == 0:
        improvement = 100.0 * (qpd_zero_loss.item() - current_loss) / qpd_zero_loss.item()
        print(f'Step {step:03d}/{QPD_SPATIAL_CAPACITY_STEPS} | loss={current_loss:.6f} | improvement vs zero={improvement:+.2f}%')

best_qpd_spatial_improvement = 100.0 * (qpd_zero_loss.item() - best_qpd_spatial_loss) / qpd_zero_loss.item()
print('\nZero-v:', f'{qpd_zero_loss.item():.6f}')
print('Pure patchwise quantum:', f'{best_qpd_loss:.6f}')
print('Quantum + spatial:', f'{best_qpd_spatial_loss:.6f}')
print('Pure quantum improvement:', f'{best_improvement:.2f}%')
print('Quantum + spatial improvement:', f'{best_qpd_spatial_improvement:.2f}%')

Step 001/150 | loss=0.895830 | improvement vs zero=-0.00%
Step 025/150 | loss=0.886239 | improvement vs zero=+1.07%
Step 050/150 | loss=0.880804 | improvement vs zero=+1.67%
Step 075/150 | loss=0.867749 | improvement vs zero=+3.13%
Step 100/150 | loss=0.847471 | improvement vs zero=+5.39%
Step 125/150 | loss=0.833703 | improvement vs zero=+6.93%
Step 150/150 | loss=0.822441 | improvement vs zero=+8.19%

Zero-v: 0.895792
Pure patchwise quantum: 0.869687
Quantum + spatial: 0.822441
Pure quantum improvement: 2.91%
Quantum + spatial improvement: 8.19%


### Fixed-Batch Capacity Comparison

The pure patchwise model and the spatial-mixer variant are evaluated using the same four jets and fixed diffusion conditions. The pure PQC improves the loss over the zero-velocity predictor by 2.91%, while adding restricted spatial communication increases the improvement to 8.19%.